# Model Experiment


## 1. Imports and Setup


In [1]:
import torch
import torch.nn as nn
from torchvision.datasets import STL10

from model import AlexNet
from torch.optim import AdamW
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

import subprocess
from pathlib import Path

In [2]:
PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"],
        text=True
    ).strip()
)

print(PROJECT_ROOT)

/mnt/c/Users/ramom/Desktop/computer-vision-architectures


In [3]:
DATA_DIR = PROJECT_ROOT / 'data'

if not DATA_DIR.exists():
    Path.mkdir(DATA_DIR)

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

Using: cuda


## 2. Configuration


In [5]:
NUM_CLASSES = 10
BATCH_SIZE = 32

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

## 3. Model


In [6]:
model = AlexNet(num_classes=NUM_CLASSES).to(device)

## 4. Smoke Test


In [7]:
x = torch.randn(4, 3, 224, 224).to(device)

with torch.no_grad():
    y = model(x)

print(f'Input shape: {x.shape}')
print(f'Output shape: {y.shape}')

assert y.shape == (x.shape[0], NUM_CLASSES)

Input shape: torch.Size([4, 3, 224, 224])
Output shape: torch.Size([4, 10])


## 5. Dataset & DataLoaders


### 5.1 Download Dataset

In [8]:
dataset = STL10(DATA_DIR / 'stl10', download=True, split='train')

Files already downloaded and verified


### 5.2 Data Augmentation Transforms

In [9]:
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
])

### 5.3 TransformWrapper

In [10]:
class TransformWrapper(Dataset):
    def __init__(self, dataset: Dataset, transform: transforms = None):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        image, label = self.dataset[index]

        if self.transform:
            image = self.transform(image)

        return image, label

final_dataset = TransformWrapper(dataset, transform)

### 5.4 DataLoaders

In [11]:
data_loader = DataLoader(final_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

## 6. Loss & Optimizer


In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(
    params=model.parameters(), 
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

## 7. Training


### 7.1 Helping Functions

In [13]:
def train_batch(
    images: torch.Tensor,
    labels: torch.Tensor,
    model: AlexNet,
    criterion: nn.CrossEntropyLoss,
    optimizer: torch.optim.SGD
):
    model.train()
    logits = model(images)
    loss = criterion(logits, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

@torch.no_grad()
def test_batch(
    images: torch.Tensor,
    labels: torch.Tensor,
    model: AlexNet,
    criterion: nn.CrossEntropyLoss,
):
    model.eval()
    logits = model(images)
    loss = criterion(logits, labels)

    return loss.item()

### 7.2  Training Loop Overfit Test

In [14]:
for epoch in range(1, 51):
    total_loss = 0.0

    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device).long()

        loss = train_batch(
            images,
            labels,
            model,
            criterion,
            optimizer
        )

        total_loss += loss

    avg_loss = total_loss / len(data_loader)

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch}: "
            f"Loss={avg_loss:.4f}"
        )

Epoch 1: Loss=2.8301
Epoch 10: Loss=1.3235
Epoch 20: Loss=0.1384
Epoch 30: Loss=0.0453
Epoch 40: Loss=0.0614
Epoch 50: Loss=0.0334
